## Inverse Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
df1 = pd.read_csv("../../datasets/inverse_repairs.csv")

df1

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
# T-box changes coming from the relational db
print(len(df1[(df1['C_deleted'] == True)] ))
print(len(df1[(df1['C_deprecated'] == True)] ))
print(len(df1[(df1['CQ_added_exception'] == True)] ))
print(len(df1[(df1['CQ_replacement_property'] == True)] ))

- check for base statement deletions:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q42034555"
property = "http://www.wikidata.org/entity/P629"
print(isRemoved(subject, property))


In [ ]:
df1['S_deleted'] = None

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
        
    subject = row['subject']
    property = row['property']
    
    # Call isRemoved function
    removed = isRemoved(subject, property)

    # Update S_deleted column
    df1.at[index, 'S_deleted'] = removed

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedWithObj(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
#subject = "http://www.wikidata.org/entity/Q11861442"
#property = "http://www.wikidata.org/prop/direct/P1026"
#print(isRemoved(subject, property))


In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (
     (row['S_deleted'] == False) &
     (row['object'].startswith('http'))
    ):
        obj = row['object']
        
        # Call isRemovedWithObj function
        wdtStmtRemoved = isRemovedWithObj(row['subject'], row['property'], obj)

        # Update S_deleted column
        df1.at[index, 'S_deleted'] = wdtStmtRemoved
            

# Display the updated DataFrame
#print(filtered_df)

- test for the addition of the context inverse statement:

In [ ]:
df1['Sc_added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def inverseAdded(obj, req_prop):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    req_prop = req_prop.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{obj}> <{req_prop}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
#test
inverseAdded('http://www.wikidata.org/entity/Q927337','http://www.wikidata.org/prop/direct/P925')

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (
     (row['object'].startswith('http'))
    ):
        obj = row['object']
        requested_property = row['PC']

        # Call inverseAdded function
        inverse = inverseAdded(obj, requested_property)

        # Update inverseAdded column
        df1.at[index, 'Sc_added'] = inverse

# Display the updated DataFrame
#print(filtered_df)

- checking rows with no classification yet:

In [ ]:
df1[(df1['C_deleted'] == False) & 
     (df1['C_deprecated'] == False)& 
     (df1['CQ_added_exception'] == False)& 
     (df1['CQ_replacement_property'] == False)& 
     (df1['S_deleted'] == False)& 
     (df1['Sc_added'] == False)
    ]

- blank nodes came also in the set of repairs because when querying the HDT dumps they got a random id, those who didnt fall in any repair tpye should be removed because are still violations.


In [ ]:
rows_to_drop = df1[
    (df1['C_deleted'] == False) &
    (df1['C_deprecated'] == False) &
    (df1['CQ_added_exception'] == False) &
    (df1['CQ_replacement_property'] == False) &
    (df1['S_deleted'] == False) &
    (df1['Sc_added'] == False) &
    (df1['object'].str.startswith('genid'))
]

In [ ]:
df1_filtered = df1.drop(rows_to_drop.index)

In [ ]:
df1_filtered[(df1_filtered['C_deleted'] == False) & 
     (df1_filtered['C_deprecated'] == False)& 
     (df1_filtered['CQ_added_exception'] == False)& 
     (df1_filtered['CQ_replacement_property'] == False)& 
     (df1_filtered['S_deleted'] == False)& 
     (df1_filtered['Sc_added'] == False)
    ]

In [ ]:
df1_filtered.to_csv('final_inverse_repairs.csv', index=False)

- generate Venn diagram with repairs distribution:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = df1_filtered['S_deleted'] | df1_filtered['Sc_added']
df2['T-box changes'] = (
    df1_filtered['C_deleted'] | 
    df1_filtered['C_deprecated'] | 
    df1_filtered['CQ_added_exception'] | 
    df1_filtered['CQ_replacement_property']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Inverse Constraint share of repairs")

plt.show()
